# 01 - Mayo Data Preparation and Sparse-View CT Degradation

This notebook prepares the Mayo dataset for sparse-view CT experiments. It only loads and preprocesses the images, applies the forward CT operator, adds Gaussian measurement noise, and saves the resulting tensors. No reconstruction algorithm is executed here.

## Environment, Paths, and Fixed Parameters

The notebook is designed for Google Colab. Google Drive is mounted first, then the course IPPy package is added to `sys.path`. The Drive layout is assumed to be correct and already populated with the Mayo dataset and IPPy package. ASTRA is installed in the runtime because IPPy's CT projector depends on it for sinogram generation.

In [ ]:
!pip install astra-toolbox

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
MAYO_DIR = PROJECT_ROOT / "Mayo"
TRAIN_DIR = MAYO_DIR / "train"
TEST_DIR = MAYO_DIR / "test"
IPPY_DIR = PROJECT_ROOT / "IPPy"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, utilities
from IPPy.utilities import load_image, normalize

SEED = 42
IMAGE_SIZE = 256
ANGLE_COUNTS = (180, 90, 60, 45)
DETECTOR_SIZE = 256
GEOMETRY = "parallel"
NOISE_LEVEL = 0.005
SHARD_SIZE = 8

torch.manual_seed(SEED)
np.random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Mayo directory:", MAYO_DIR)
print("IPPy directory:", IPPY_DIR)
print("Output directory:", OUTPUT_DIR)
print("Angle counts:", ANGLE_COUNTS)
print("Noise level:", NOISE_LEVEL)

## Dataset Loading and Preprocessing

Mayo slices are loaded with the IPPy utility used in the course material. Each image is converted to the 4D PyTorch format `[batch, channels, height, width]` before resizing, because `torch.nn.functional.interpolate` expects 4D tensors for 2D images. After resizing, `IPPy.utilities.normalize` keeps the image scale consistent with the CT exercises.

In [ ]:
def collect_png_paths(split_dir: Path) -> list[Path]:
    return sorted(split_dir.glob("*/*.png"))


def ensure_4d_grayscale(image: torch.Tensor) -> torch.Tensor:
    image = torch.as_tensor(image).float()

    if image.ndim == 2:
        image = image.unsqueeze(0).unsqueeze(0)
    elif image.ndim == 3:
        image = image.unsqueeze(0) if image.shape[0] == 1 else image.permute(2, 0, 1).unsqueeze(0)

    return image


def load_and_preprocess_image(image_path: Path) -> torch.Tensor:
    image = load_image(str(image_path))
    image = ensure_4d_grayscale(image)
    image = F.interpolate(
        image,
        size=(IMAGE_SIZE, IMAGE_SIZE),
        mode="bilinear",
        align_corners=False,
    )
    return normalize(image).clamp(0.0, 1.0).to(torch.float32)


def load_clean_batch(paths: list[Path]) -> torch.Tensor:
    return torch.cat([load_and_preprocess_image(path) for path in paths], dim=0)


split_paths = {
    "train": collect_png_paths(TRAIN_DIR),
    "test": collect_png_paths(TEST_DIR),
}

for split_name, paths in split_paths.items():
    print(f"{split_name}: {len(paths)} images")

sample_clean = load_and_preprocess_image(split_paths["train"][0])
print("Sample clean shape:", tuple(sample_clean.shape))
print("Sample clean range:", float(sample_clean.min()), float(sample_clean.max()))

## Forward CT Model and Export Functions

The forward model follows the CT reconstruction exercise: `IPPy.operators.CTProjector` computes parallel-beam sinograms with a fixed detector size. Four projectors are created, one for each sparse-view configuration. The noisy measurements are saved without clamping because sinograms are projection data, not image intensities.

In [ ]:
projectors = {
    str(n_views): operators.CTProjector(
        img_shape=(IMAGE_SIZE, IMAGE_SIZE),
        angles=np.linspace(0.0, np.pi, n_views),
        det_size=DETECTOR_SIZE,
        geometry=GEOMETRY,
    )
    for n_views in ANGLE_COUNTS
}


@torch.no_grad()
def make_noisy_sinograms(clean_batch: torch.Tensor) -> tuple[dict[str, torch.Tensor], dict[str, float]]:
    sinograms = {}
    relative_noise = {}

    for key, projector in projectors.items():
        noisy_measurements = []
        sample_noise_levels = []

        for image in clean_batch:
            x = image.unsqueeze(0)
            y = projector(x)
            noise = utilities.gaussian_noise(y, noise_level=NOISE_LEVEL)

            noisy_measurements.append((y + noise).detach().cpu().to(torch.float32))
            sample_noise_levels.append(float(torch.norm(noise.detach().cpu()) / torch.norm(y.detach().cpu())))

        sinograms[key] = torch.cat(noisy_measurements, dim=0)
        relative_noise[key] = float(np.mean(sample_noise_levels))

    return sinograms, relative_noise


def save_split_shards(split_name: str, paths: list[Path]) -> list[dict]:
    shard_records = []
    patient_paths = {}

    for path in paths:
        patient_id = path.parent.name
        patient_paths.setdefault(patient_id, []).append(path)

    patient_shards = [
        (patient_id, start)
        for patient_id, patient_image_paths in patient_paths.items()
        for start in range(0, len(patient_image_paths), SHARD_SIZE)
    ]

    for patient_id, start in tqdm(
        patient_shards,
        total=len(patient_shards),
        desc=f"{split_name} shards",
    ):
        patient_image_paths = patient_paths[patient_id]
        patient_shard_index = start // SHARD_SIZE
        batch_paths = patient_image_paths[start:start + SHARD_SIZE]
        clean_batch = load_clean_batch(batch_paths)
        sinograms, relative_noise = make_noisy_sinograms(clean_batch)

        shard_name = f"{patient_id}_shard_{patient_shard_index:04d}.pt"
        shard_path = OUTPUT_DIR / split_name / patient_id / shard_name
        shard_path.parent.mkdir(parents=True, exist_ok=True)
        source_paths = [str(path) for path in batch_paths]

        payload = {
            "clean": clean_batch.detach().cpu().to(torch.float32),
            "sinograms": sinograms,
            "source_paths": source_paths,
            "metadata": {
                "split": split_name,
                "patient_id": patient_id,
                "patient_shard_index": patient_shard_index,
                "num_samples": len(batch_paths),
                "image_size": IMAGE_SIZE,
                "angle_counts": list(ANGLE_COUNTS),
                "detector_size": DETECTOR_SIZE,
                "geometry": GEOMETRY,
                "noise_level": NOISE_LEVEL,
                "relative_noise": relative_noise,
            },
        }
        torch.save(payload, shard_path)

        shard_records.append(
            {
                "path": str(shard_path.relative_to(OUTPUT_DIR)),
                "patient_id": patient_id,
                "patient_shard_index": patient_shard_index,
                "num_samples": len(batch_paths),
                "source_paths": source_paths,
                "relative_noise": relative_noise,
            }
        )

    return shard_records


sample_sinograms, sample_noise = make_noisy_sinograms(sample_clean)
for key, value in sample_sinograms.items():
    print(f"{key} views: {tuple(value.shape)} | relative noise {sample_noise[key]:.6f}")

## Generate the Processed Dataset

This cell runs the full data preparation step for both splits. The exported `.pt` shards recreate the Mayo split/patient directory structure, and each file name includes the patient identifier. Every shard contains clean images, noisy sparse-view sinograms, source paths, and metadata. The manifest records the configuration so downstream notebooks can reuse the same degraded data.

In [ ]:
start_time = time.time()

manifest = {
    "project_root": str(PROJECT_ROOT),
    "mayo_dir": str(MAYO_DIR),
    "output_dir": str(OUTPUT_DIR),
    "image_size": IMAGE_SIZE,
    "angle_counts": list(ANGLE_COUNTS),
    "detector_size": DETECTOR_SIZE,
    "geometry": GEOMETRY,
    "noise_level": NOISE_LEVEL,
    "shard_size": SHARD_SIZE,
    "normalization": "IPPy.utilities.normalize after IPPy.utilities.load_image and resize",
    "splits": {},
}

for split_name, paths in split_paths.items():
    shard_records = save_split_shards(split_name, paths)
    manifest["splits"][split_name] = {
        "num_images": len(paths),
        "num_shards": len(shard_records),
        "shards": shard_records,
    }

manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved manifest:", manifest_path)
print(f"Elapsed time: {(time.time() - start_time) / 60:.2f} minutes")

## Visual Sanity Check and Output Summary

The final cell reloads the first saved shard and displays one clean slice with its four noisy sinograms. This is only an inspection of the generated measurements and does not apply any inverse or reconstruction method.

In [ ]:
preview_split = "train"
preview_shard_path = OUTPUT_DIR / manifest["splits"][preview_split]["shards"][0]["path"]
preview_payload = torch.load(preview_shard_path, map_location="cpu")

clean_preview = preview_payload["clean"][0, 0]
sinogram_previews = preview_payload["sinograms"]

fig, axes = plt.subplots(1, 1 + len(ANGLE_COUNTS), figsize=(4 * (1 + len(ANGLE_COUNTS)), 4), constrained_layout=True)
axes[0].imshow(clean_preview, cmap="gray", vmin=0.0, vmax=1.0)
axes[0].set_title("Clean 256x256")
axes[0].axis("off")

for ax, n_views in zip(axes[1:], ANGLE_COUNTS):
    key = str(n_views)
    ax.imshow(sinogram_previews[key][0, 0], cmap="gray", aspect="auto")
    ax.set_title(f"{key} views")
    ax.set_xlabel("Detector")
    ax.set_ylabel("Angle")

plt.show()

print("Processed dataset root:", OUTPUT_DIR)
print("Manifest:", OUTPUT_DIR / "manifest.json")
for split_name, split_info in manifest["splits"].items():
    print(f"{split_name}: {split_info['num_images']} images, {split_info['num_shards']} shards")

## Loading Contract for Downstream Notebooks

The following cell defines the data interface that the reconstruction and comparison notebooks should use. Each shard is a PyTorch dictionary containing clean ground-truth images, one noisy sinogram tensor for each sparse-view setting, source file paths, and metadata. This contract avoids regenerating degraded data in later notebooks.

In [ ]:
contract_split = "train"
contract_shard_path = OUTPUT_DIR / manifest["splits"][contract_split]["shards"][0]["path"]
contract_payload = torch.load(contract_shard_path, map_location="cpu")

clean = contract_payload["clean"]
sinogram_180 = contract_payload["sinograms"]["180"]
sinogram_90 = contract_payload["sinograms"]["90"]
sinogram_60 = contract_payload["sinograms"]["60"]
sinogram_45 = contract_payload["sinograms"]["45"]
source_paths = contract_payload["source_paths"]
metadata = contract_payload["metadata"]

print("Shard:", contract_shard_path)
print("clean:", tuple(clean.shape))
print("sinogram_180:", tuple(sinogram_180.shape))
print("sinogram_90:", tuple(sinogram_90.shape))
print("sinogram_60:", tuple(sinogram_60.shape))
print("sinogram_45:", tuple(sinogram_45.shape))
print("patient_id:", metadata["patient_id"])
print("first source path:", source_paths[0])